In [2]:
import numpy as np
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
path = "./Dataset"
target_size = (224, 224)

In [ ]:
insect_classes = ["Butterfly", "Dragonfly", "Grasshopper", "Ladybird", "Mosquito"]
dataset = []
labels = []
name = []
for insect_class in insect_classes:
    class_path = Path(path) / insect_class
    for img_path in class_path.glob("*.jpg"):
        img = Image.open(img_path).convert("RGB")   
        img = img.resize(target_size)
        dataset.append(np.array(img))
        labels.append(insect_class)
        name.append(img_path.name)
dataset = np.array(dataset)
labels = np.array(labels)
name = np.array(name)

In [ ]:

axs = plt.figure(figsize=(10, 10)).subplots(6, 5)
axs = axs.flatten()
for i in range(30):
    n = np.random.randint(0, len(dataset))
    axs[i].imshow(dataset[n])
    axs[i].set_title(labels[n])
    axs[i].axis("off")


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, name_train, name_test = train_test_split(
    dataset, labels, name, test_size=0.2, random_state=42
)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape, name_train.shape, name_test.shape

In [3]:
import torch
import cv2
import numpy as np
import requests
from PIL import Image
from torchvision import models, transforms
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
# 3. Image Preprocessing Pipeline
def preprocess_image(img):
    rgb_img = np.float32(img) / 255
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((224, 224)),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Create input tensor and ensure it's resized for the overlay later
    input_tensor = transform(img).unsqueeze(0)
    rgb_img_resized = cv2.resize(rgb_img, (224, 224))
    return input_tensor, rgb_img_resized

# 4. Run Grad-CAM
# Replace 'your_image.jpg' with your local file path
input_tensor, rgb_img = preprocess_image(dataset[0])  # Using the first image from the dataset for demonstration

In [ ]:
print(dataset.shape)
print(X_train.shape)
print(X_test.shape)
print(input_tensor.shape)
print(rgb_img.shape)

In [8]:
class GrayscaleCAMItem:
    def __init__(self, grayscale_cam, name, label, original_img=None):
        self.grayscale_cam = np.asarray(grayscale_cam)
        self.name = name
        self.label = label
        self.original_img = original_img  # Store the original image, if provided

    def to_heatmap(self):
        return cv2.applyColorMap(np.uint8(255 * self.grayscale_cam), cv2.COLORMAP_JET)[:,:,::-1]

    def to_mask(self, threshold=0.5):
        return np.where(self.grayscale_cam > threshold, 1, 0)
    
    def to_overlay(self):
        if self.original_img is None:
            raise ValueError("Original image must be provided for overlay.")
        overlay = show_cam_on_image(self.original_img, self.grayscale_cam, use_rgb=True)
        return overlay
    
    def to_applied_mask(self, threshold=0.5):
        if self.original_img is None:
            raise ValueError("Original image must be provided for applied mask.")
        mask = self.to_mask(threshold)
        applied_mask = self.original_img * mask.reshape(224, 224, 1)
        return applied_mask

RESNET152

In [ ]:
def resnet152_gradcam(input_tensor, rgb_img, threshold=0.5):
    # 1. Load a pre-trained ResNet-152 model
    model = models.resnet152(weights=models.ResNet152_Weights.IMAGENET1K_V1)
    model.eval()

    # 2. Target the last convolutional layer
    # For ResNet-152, this is the very last bottleneck block
    target_layers = [model.layer4[-1]]

    # Gradient-weighted Class Activation Mapping (Grad-CAM) is a technique used to visualize the regions of an input
    # image that are most important for a convolutional neural network's (CNN) decision-making process.
    cam = GradCAM(model=model, target_layers=target_layers)

    # If targets=None, it will show the CAM for the highest scoring category
    # The output is a grayscale heatmap of shape [1, 224, 224] that can be overlaid on the original image
    # and values are in the range [0, 1] where higher values indicate more important regions for the model's prediction.
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)  # You can specify a target class index if desired
    mask = np.where(grayscale_cam > threshold, 1, 0)[0]  # Thresholding the CAM for better visualization
    # 5. Overlay and Save
    # We take [0, :] because we only passed one image (batch size 1)
    visualization = show_cam_on_image(rgb_img, grayscale_cam[0, :], use_rgb=True)
    heatmap = cv2.applyColorMap(np.uint8(255 * grayscale_cam[0, :]), cv2.COLORMAP_JET)
    # cam_resnet = visualization
    return heatmap[:,:,::-1], visualization, mask
# plt.imshow(visualization)
# plt.show()
# plt.imshow(rgb_img * mask.reshape(224, 224,1))  # Apply the mask to the original image

In [ ]:
tensor, rgb_img = preprocess_image(dataset[1])  # Using the first image from the dataset for demonstration
heatmap1, visualization1, mask1 = resnet152_gradcam(tensor, rgb_img, threshold=0.5)
axs = plt.figure(figsize=(10, 10)).subplots(1, 5)
axs = axs.flatten()
axs[0].imshow(rgb_img)
axs[0].set_title("Original Image")
axs[0].axis("off")
axs[1].imshow(heatmap1)
axs[1].set_title("Heatmap")
axs[1].axis("off")
axs[2].imshow(visualization1)
axs[2].set_title("Visualization")
axs[2].axis("off")
axs[3].imshow(mask1, cmap="gray")
axs[3].set_title("Mask")
axs[3].axis("off")
axs[4].imshow(rgb_img * mask1.reshape(224, 224,1))  # Apply the mask to the original image
axs[4].set_title("Masked Image")
axs[4].axis("off")



In [ ]:
def resnet(images, labels, name, path):
    model = models.resnet152(weights=models.ResNet152_Weights.IMAGENET1K_V1)
    model.eval()
    target_layers = [model.layer4[-1]]
    cam = GradCAM(model=model, target_layers=target_layers)
    for i in range(images.shape[0]):
        input_tensor, _ = preprocess_image(images[i])
        grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0]
        cam_item = GrayscaleCAMItem(grayscale_cam, name[i], labels[i])
        save_dir = Path(path)
        save_dir.mkdir(parents=True, exist_ok=True)
        np.save(save_dir / f"{i}", cam_item)
        if i % 100 == 0:
            print(f"Processed {i}/{images.shape[0]} images")   

In [ ]:
resnet(X_train[:20], y_train[:20], name_train[:20], path="./Heatmap_train/resnet")

In [ ]:
#---------- LONG EXECUTION -----------------
def long_execution1():
    resnet(X_train, y_train, name_train, path="./Heatmap_train/resnet")
    resnet(X_test, y_test, name_test, path="./Heatmap_test/resnet")

In [ ]:
# Load saved grayscale CAMs from ./Heatmap/resnet/Butterfly (fallback to lowercase)
load_dir = Path("./Heatmap_test/resnet") / "Butterfly"
if not load_dir.exists():
    load_dir = Path("./Heatmap_test/resnet") / "butterfly"

npy_files = sorted(load_dir.glob("*.npy"))
cams = {p.stem: np.load(p) for p in npy_files}

print(f"Loaded {len(cams)} CAMs from: {load_dir}")
for name, cam in list(cams.items())[:10]:
    print(name, cam.shape)

# Example: visualize the first loaded CAM (handle possible [1,H,W] shape)
if cams:
    first_key = next(iter(cams))
    cam = cams[first_key]
    # if cam.ndim == 3 and cam.shape[0] == 1:
    #     cam = cam[0]
    plt.figure(figsize=(6,6))
    plt.imshow(cam, cmap='jet')
    plt.title(first_key)
    plt.axis('off')

Vision Transformer

In [ ]:
# 1. Define the Reshape Transform for ViT
def vit_reshape_transform(tensor, height=14, width=14):
    # The input 'tensor' has shape [Batch, Num_Tokens, Embedding_Dim]
    # For ViT-B/16, it is [1, 197, 768]
    
    # Discard the CLS token (the first one)
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))

    # Bring the channels to the second dimension for PyTorch compatibility
    # Shape becomes [Batch, Channels, Height, Width]
    result = result.permute(0, 3, 1, 2)
    return result

def viT_gradcam(input_tensor, rgb_img, threshold=0.5):
    # 2. Load ViT Model
    model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1).eval()

    # 3. Target the last Norm layer in the last Transformer block
    target_layers = [model.encoder.layers[-1].ln_1]

    # 4. Initialize Grad-CAM with the reshape_transform
    cam = GradCAM(model=model, 
                target_layers=target_layers, 
                reshape_transform=vit_reshape_transform)

    # 5. Process Image
    # (Assuming 'input_tensor' and 'rgb_img' are prepared as in previous steps)
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)
    mask = np.where(grayscale_cam> threshold, 1, 0)[0]  # Thresholding the CAM for better visualization
    # 5. Overlay and Save
    # We take [0, :] because we only passed one image (batch size 1)
    visualization = show_cam_on_image(rgb_img, grayscale_cam[0, :], use_rgb=True)
    heatmap = cv2.applyColorMap(np.uint8(255 * grayscale_cam[0, :]), cv2.COLORMAP_JET)
    # cam_vit = visualization
    return heatmap[:,:,::-1], visualization, mask
# plt.imshow(visualization)
# plt.show()
# plt.imshow(rgb_img * mask.reshape(224, 224,1))  # Apply the mask to the original image

In [ ]:
tensor, rgb_img = preprocess_image(dataset[1])  # Using the first image from the dataset for demonstration
heatmap2, visualization2, mask2 = viT_gradcam(tensor, rgb_img, threshold=0.5)
axs = plt.figure(figsize=(10, 10)).subplots(1, 5)
axs = axs.flatten()
axs[0].imshow(rgb_img)
axs[0].set_title("Original Image")
axs[0].axis("off")
axs[1].imshow(heatmap2)
axs[1].set_title("Heatmap")
axs[1].axis("off")
axs[2].imshow(visualization2)
axs[2].set_title("Visualization")
axs[2].axis("off")
axs[3].imshow(mask2, cmap="gray")
axs[3].set_title("Mask")
axs[3].axis("off")
axs[4].imshow(rgb_img * mask2.reshape(224, 224,1))  # Apply the mask to the original image
axs[4].set_title("Masked Image")
axs[4].axis("off")

In [ ]:
def vit_reshape_transform(tensor, height=14, width=14):
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    result = result.permute(0, 3, 1, 2)
    return result

def viT(images, labels, name, path):
    model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1).eval()
    target_layers = [model.encoder.layers[-1].ln_1]
    cam = GradCAM(model=model, 
                target_layers=target_layers, 
                reshape_transform=vit_reshape_transform)
    
    for i in range(images.shape[0]):
        input_tensor, _ = preprocess_image(images[i])
        grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0]
        cam_item = GrayscaleCAMItem(grayscale_cam, name[i], labels[i])
        save_dir = Path(path)
        save_dir.mkdir(parents=True, exist_ok=True)
        np.save(save_dir / f"{i}", cam_item)
        if i % 100 == 0:
            print(f"Processed {i}/{images.shape[0]} images")
    

In [ ]:
viT(X_train[:20], y_train[:20], name_train[:20], path="./Heatmap_train/viT")

In [ ]:
def long_execution2():
    viT(X_train, y_train, name_train, path="./Heatmap_train/viT")
    viT(X_test, y_test, name_test, path="./Heatmap_test/viT")

Swin Transformer

In [ ]:
# 2. Define the Reshape Transform for Swin
def swin_reshape_transform(tensor, height=7, width=7):
    # tensor shape is [1, 49, 1024]
    # We want to reshape it to [1, 7, 7, 1024]
    
    # Use -1 for the channel dimension so it "auto-fills" the 1024
    result = tensor.reshape(tensor.size(0), height, width, -1)
    
    # Now move channels to the front for Grad-CAM: [1, 1024, 7, 7]
    result = result.permute(0, 3, 1, 2)
    return result

def swinT_gradcam(input_tensor, rgb_img, threshold=0.5):
    # 1. Load the pre-trained Swin Transformer
    model = models.swin_b(weights=models.Swin_B_Weights.IMAGENET1K_V1).eval()
    # 3. Select the target layer
    # We target the very last layer of the feature extractor
    target_layers = [model.features[-1]]

    # 4. Initialize Grad-CAM
    cam = GradCAM(model=model, 
                target_layers=target_layers, 
                reshape_transform=swin_reshape_transform)

    # 5. Run the CAM
    # (Assuming input_tensor is already on .cuda())
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)
    mask = np.where(grayscale_cam> threshold, 1, 0)[0]  # Thresholding the CAM for better visualization
    # 5. Overlay and Save
    # We take [0, :] because we only passed one image (batch size 1)
    visualization = show_cam_on_image(rgb_img, grayscale_cam[0, :], use_rgb=True)
    heatmap = cv2.applyColorMap(np.uint8(255 * grayscale_cam[0, :]), cv2.COLORMAP_JET)
    # cam_swint = visualization
    return heatmap[:,:,::-1], visualization, mask
# plt.imshow(visualization)
# plt.show()
# plt.imshow(rgb_img * mask.reshape(224, 224,1))  # Apply the mask to the original image

In [ ]:
tensor, rgb_img = preprocess_image(dataset[1])  # Using the first image from the dataset for demonstration
heatmap3, visualization3, mask3 = swinT_gradcam(tensor, rgb_img, threshold=0.5)
axs = plt.figure(figsize=(10, 10)).subplots(1, 5)
axs = axs.flatten()
axs[0].imshow(rgb_img)
axs[0].set_title("Original Image")
axs[0].axis("off")
axs[1].imshow(heatmap3)
axs[1].set_title("Heatmap")
axs[1].axis("off")
axs[2].imshow(visualization3)
axs[2].set_title("Visualization")
axs[2].axis("off")
axs[3].imshow(mask3, cmap="gray")
axs[3].set_title("Mask")
axs[3].axis("off")
axs[4].imshow(rgb_img * mask3.reshape(224, 224,1))  # Apply the mask to the original image
axs[4].set_title("Masked Image")
axs[4].axis("off")

In [ ]:
def swin_reshape_transform(tensor, height=7, width=7):
    result = tensor.reshape(tensor.size(0), height, width, -1)
    result = result.permute(0, 3, 1, 2)
    return result

def swinT(images, labels, name, path):
    model = models.swin_b(weights=models.Swin_B_Weights.IMAGENET1K_V1).eval()
    target_layers = [model.features[-1]]
    cam = GradCAM(model=model, 
                target_layers=target_layers, 
                reshape_transform=swin_reshape_transform)
    
    for i in range(images.shape[0]):
        input_tensor, _ = preprocess_image(images[i])
        grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0]
        cam_item = GrayscaleCAMItem(grayscale_cam, name[i], labels[i])
        save_dir = Path(path)
        save_dir.mkdir(parents=True, exist_ok=True)
        np.save(save_dir / f"{i}", cam_item)
        if i % 100 == 0:
            print(f"Processed {i}/{images.shape[0]} images")

In [ ]:
swinT(X_train[:20], y_train[:20], name_train[:20], path="./Heatmap_train/swinT")

In [ ]:
def long_execution3():
    swinT(X_train, y_train, name_train, path="./Heatmap_train/swinT")
    swinT(X_test, y_test, name_test, path="./Heatmap_test/swinT")

In [ ]:

def visualize_gradcam_on_dataset(img, gradcam_function):
    #img = dataset[2] 
    input_tensor, rgb_img = preprocess_image(img)
    heatmap, cam_resnet, mask_resnet = gradcam_function(input_tensor, rgb_img, threshold=0.5)
      # Convert BGR to RGB for correct visualization

    axs = plt.figure(figsize=(15, 15)).subplots(1, 5)
    axs = axs.flatten()
    axs[0].imshow(img)
    axs[0].set_title("Original Image")
    axs[0].axis("off")

    axs[1].imshow(heatmap, cmap='jet')
    axs[1].set_title(f"{gradcam_function.__name__}")
    axs[1].axis("off")

    axs[2].imshow(cam_resnet)
    axs[2].set_title("Overlay")
    axs[2].axis("off")

    axs[3].imshow(mask_resnet, cmap='gray')
    axs[3].set_title("Mask")
    axs[3].axis("off")

    axs[4].imshow(rgb_img * mask_resnet.reshape(224, 224, 1))
    axs[4].set_title("Masked Image")
    axs[4].axis("off")
    



In [ ]:
visualize_gradcam_on_dataset(dataset[1], resnet152_gradcam)
visualize_gradcam_on_dataset(dataset[1], viT_gradcam)
visualize_gradcam_on_dataset(dataset[1], swinT_gradcam)

IFCNN


In [ ]:
# ---------------------------------------------------------------------------
# IFCNN — Attention-Selection Mechanism (fixed)
# Correzioni rispetto alla versione originale:
#   1. min/max invertiti in attention_rec_logic
#   2. eigen_smoothing riscritta per operare su un singolo sample [C, H, W]
#   3. forward gestisce i 3 CAM come lista e applica tied weights correttamente
# ---------------------------------------------------------------------------

import torch
import math
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


class ConvBlock(nn.Module):
    def __init__(self, inplane, outplane):
        super().__init__()
        self.padding = (1, 1, 1, 1)
        self.conv = nn.Conv2d(inplane, outplane, kernel_size=3, padding=0,
                              stride=1, bias=False)
        self.bn   = nn.BatchNorm2d(outplane)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = F.pad(x, self.padding, 'replicate')
        out = self.conv(out)
        out = self.bn(out)
        out = self.relu(out)
        return out


class IFCNN(nn.Module):
    def __init__(self, resnet, w0=0.7, w1=0.3):
        super().__init__()
        self.w0 = w0
        self.w1 = w1

        # CONV2: 64->64  (tied weights tra i 3 input, cioè stesso layer riusato)
        self.conv2 = ConvBlock(64, 64)

        # CONV3: 64->128  (dal diagramma Fig.3: Feature Maps 64 -> 128)
        self.conv3 = ConvBlock(64, 64)

        # CONV4: 128->3   (ricostruzione a 3 canali RGB)
        self.conv4 = nn.Conv2d(64, 3, kernel_size=1, padding=0,
                               stride=1, bias=True)

        # Init pesi conv learnable
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2.0 / n))

        # CONV1: prende conv1 di ResNet pretrained, stride=1, no padding
        for p in resnet.parameters():
            p.requires_grad = False
        self.conv1 = resnet.conv1
        self.conv1.stride  = (1, 1)
        self.conv1.padding = (0, 0)

    # ------------------------------------------------------------------
    # BUG 2 FIX: eigen_smoothing opera su un singolo sample [C, H, W]
    # (non su un batch). La SVD estrae la prima componente principale
    # nello spazio dei canali per ogni posizione spaziale.
    # ------------------------------------------------------------------
    def eigen_smoothing(self, f_hat: torch.Tensor) -> torch.Tensor:
        """
        Smoothing basato sulla prima componente principale (Ref 46, Eigen-CAM).
        Input:  f_hat  [B, C, H, W]
        Output: smoothed  [B, C, H, W]
        """
        B, C, H, W = f_hat.shape
        smoothed_list = []

        for b in range(B):
            # [C, H*W]  →  [H*W, C]
            feat = f_hat[b].view(C, -1).T          # [H*W, C]

            # Centratura
            mean   = feat.mean(dim=0, keepdim=True) # [1, C]
            centered = feat - mean                   # [H*W, C]

            # SVD: U [H*W, k], S [k], Vh [k, C]  dove k = min(H*W, C)
            U, S, Vh = torch.linalg.svd(centered, full_matrices=False)

            # Ricostruzione rank-1 (solo il primo componente)
            # U[:,0] * S[0] riproiettato su Vh[0,:]
            rank1 = (U[:, 0:1] * S[0]) @ Vh[0:1, :]  # [H*W, C]

            # Riaggiungo la media e riporto a [C, H, W]
            denoised = (rank1 + mean).T.view(C, H, W)
            smoothed_list.append(denoised)

        return torch.stack(smoothed_list, dim=0)  # [B, C, H, W]

    # BUG 1 FIX: intersection = min, union = max (Eq. 4)
    # ------------------------------------------------------------------
    def attention_rec_logic(self, features: list) -> torch.Tensor:
        """
        Eq. 3-4 del paper:
            ∩  = min tra i modelli  (consenso: pixel evidenziati da TUTTI)
            ∪  = max tra i modelli  (aggregazione: pixel evidenziati da ALMENO uno)
            f̂  = w0 * ∩  +  w1 * (∪ - ∩)
        """
        stacked = torch.stack(features, dim=0)   # [N_models, B, C, H, W]
 
        # BUG 1 era qui: erano invertiti
        intersection, _ = torch.min(stacked, dim=0)   # ∩
        union,        _ = torch.max(stacked, dim=0)   # ∪
 
        f_hat = self.w0 * intersection + self.w1 * (union - intersection)
 
        # Eigen-smoothing per ridurre il rumore
        f_hat = self.eigen_smoothing(f_hat)
 
        return f_hat
 
    # ------------------------------------------------------------------
    # BUG 3 FIX: forward gestisce lista di tensori e riusa correttamente
    # gli stessi layer (CONV1, CONV2) come tied weights per ogni CAM.
    # ------------------------------------------------------------------
    def forward(self, *tensors):
        """
        tensors: N immagini/CAM in input (nel paper N=3: ResNet, ViT, Swin-T).
        Ogni tensore ha shape [B, 3, H, W].
        """
        # --- Feature Extraction (CONV1 + CONV2, tied weights) ---
        outs = []
        for t in tensors:
            # Padding replicativo prima di CONV1 (kernel 7×7 → pad 3)
            x = F.pad(t, (3, 3, 3, 3), mode='replicate')
            x = self.conv1(x)     # [B, 64, H, W]
            x = self.conv2(x)     # [B, 64, H, W]
            outs.append(x)
 
        # --- Feature Fusion (Attention-Selection) ---
        fused = self.attention_rec_logic(outs)   # [B, 64, H, W]
 
        # --- Feature Reconstruction (CONV3 + CONV4) ---
        out = self.conv3(fused)   # [B, 128, H, W]
        out = self.conv4(out)     # [B,   3, H, W]
        return out
 
 
def myIFCNN(w0=0.7, w1=0.3):
    resnet = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V1)
    model  = IFCNN(resnet, w0=w0, w1=w1)
    return model
 


In [ ]:
# --- Quick sanity check ---
model = myIFCNN(w0=0.7, w1=0.3)
model.load_state_dict(torch.load('weights/IFCNN-SUM.pth', map_location=torch.device('cpu')))
model.eval()

# Simula 3 CAM in input (ResNet, ViT, Swin-T), batch=2, 224×224
# cam1 = torch.randn(2, 3, 224, 224)
# cam2 = torch.randn(2, 3, 224, 224)
# cam3 = torch.randn(2, 3, 224, 224)

# 2. Cast to float32
# t_heat1 = torch.from_numpy(heatmap1.copy()).float()
# t_heat2 = torch.from_numpy(heatmap2.copy()).float()
# t_heat3 = torch.from_numpy(heatmap3.copy()).float()

t_resnet = torch.tensor(heatmap1.copy(), dtype=torch.float32).permute(2, 0, 1).unsqueeze(0) 
t_vit = torch.tensor(heatmap2.copy(), dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)
t_swint = torch.tensor(heatmap3.copy(), dtype=torch.float32).permute(2, 0, 1).unsqueeze(0) 

# 3. (Optional but likely needed) Ensure the shape is [Batch, Channels, Height, Width]
# If your heatmaps are [Height, Width] or [Height, Width, Channels], 
# you will need to unsqueeze or permute them here before passing them to the model.

with torch.no_grad():
    out = model(t_resnet, t_vit, t_swint)
# with torch.no_grad():
#     out = model(heatmap1.copy(), heatmap2.copy(), heatmap3.copy())

print(f"Input shapes : {t_resnet.shape} × 3")
print(f"Output shape : {out.shape}")   # atteso: [2, 3, 224, 224]
print("OK — nessun errore di shape.")

In [4]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [ ]:
def show_ifcnn_heatmap(
    original_img, #: torch.Tensor,   # [3, H, W] float in [0,1]
    ifcnn_output: torch.Tensor,   # [B, 3, H, W] output grezzo di IFCNN
    sample_idx: int = 0,
    threshold: float = 0.4,
    alpha: float = 0.5,
    colormap: str = 'jet',
):
    # 1. Prendi il sample e collassa i 3 canali in uno scalare per pixel
    cam = ifcnn_output[sample_idx]           # [3, H, W]
    # plt.imshow(cam.permute(1, 2, 0).detach().cpu().numpy())
    cam_gray = cam.mean(dim=0)                  # [H, W]

    # 2. Normalizza in [0, 1]
    cam_gray = cam_gray - cam_gray.min()
    cam_gray = cam_gray / (cam_gray.max() + 1e-8)
    cam_np = cam_gray.detach().cpu().numpy()    # [H, W]
    # plt.imshow(cam_np, cmap=colormap)
    # 3. Applica colormap → immagine RGB
    heatmap = cm.get_cmap(colormap)(cam_np)[:, :, :3]  # [H, W, 3]

    # 4. Overlay sull'immagine originale
    img_np = original_img #.permute(1, 2, 0).detach().cpu().numpy()
    overlay = (1 - alpha) * img_np + alpha * heatmap

    # 5. Maschera binaria (dove la CAM supera la soglia)
    mask = (cam_np >= threshold).astype(np.float32)
    masked_img = img_np * mask[:, :, np.newaxis]

    # 6. Plot (single original image in the first column)
    fig = plt.figure(figsize=(12, 10))
    gs = fig.add_gridspec(4, 4)

    # First column: one single original image spanning all rows
    ax0 = fig.add_subplot(gs[:, 0])
    ax0.imshow(img_np.clip(0, 1))
    ax0.set_title("Originale")
    ax0.axis("off")

    # Row 1: ResNet
    ax = fig.add_subplot(gs[0, 1]); ax.imshow(heatmap1); ax.set_title("resnet152_gradcam"); ax.axis("off")
    ax = fig.add_subplot(gs[0, 2]); ax.imshow(visualization1); ax.set_title("Overlay"); ax.axis("off")
    ax = fig.add_subplot(gs[0, 3]); ax.imshow(rgb_img * mask1.reshape(224, 224, 1)); ax.set_title("Key areas (thr=0.5)"); ax.axis("off")

    # Row 2: ViT
    ax = fig.add_subplot(gs[1, 1]); ax.imshow(heatmap2); ax.set_title("viT_gradcam"); ax.axis("off")
    ax = fig.add_subplot(gs[1, 2]); ax.imshow(visualization2); ax.set_title("Overlay"); ax.axis("off")
    ax = fig.add_subplot(gs[1, 3]); ax.imshow(rgb_img * mask2.reshape(224, 224, 1)); ax.set_title("Key areas (thr=0.5)"); ax.axis("off")

    # Row 3: SwinT
    ax = fig.add_subplot(gs[2, 1]); ax.imshow(heatmap3); ax.set_title("swinT_gradcam"); ax.axis("off")
    ax = fig.add_subplot(gs[2, 2]); ax.imshow(visualization3); ax.set_title("Overlay"); ax.axis("off")
    ax = fig.add_subplot(gs[2, 3]); ax.imshow(rgb_img * mask3.reshape(224, 224, 1)); ax.set_title("Key areas (thr=0.5)"); ax.axis("off")

    # Row 4: IFCNN
    ax = fig.add_subplot(gs[3, 1]); ax.imshow(cam_np, cmap=colormap); ax.set_title("IFCNN_gradcam"); ax.axis("off")
    ax = fig.add_subplot(gs[3, 2]); ax.imshow(overlay.clip(0, 1)); ax.set_title("Overlay"); ax.axis("off")
    ax = fig.add_subplot(gs[3, 3]); ax.imshow(masked_img.clip(0, 1)); ax.set_title(f"Key areas (thr={threshold})"); ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# with torch.no_grad():
#     rec_cam = model(t_resnet, t_vit, t_swint)   # [B, 3, H, W]
# print(f"IFCNN output shape: {rec_cam.shape}")  # atteso: [1, 3, 224, 224]

show_ifcnn_heatmap(rgb_img, out, sample_idx=0)

In [ ]:
def show_ifcnn_heatmap(ifcnn_output):
    cam = ifcnn_output[0]
    cam_gray = cam.mean(dim=0)
    cam_gray = cam_gray - cam_gray.min()
    cam_gray = cam_gray / (cam_gray.max() + 1e-8)
    cam_np = cam_gray.detach().cpu().numpy()
    return cam_np

# def from_greyscale_to_heatmap_rgb(cam_gray):
#    return cv2.applyColorMap(np.uint8(255 * cam_gray), cv2.COLORMAP_JET)

# def upload_from_memory():
#     maps = ["resnet", "swinT", "viT"]
#     greyscale_heatmap = []

#     for m in maps:
#         load_dir = Path(f"./Heatmap_train/{m}")
#         npy_files = sorted(load_dir.glob("*.npy"))

#         cam_stack = []
#         for p in npy_files:
#             cam_item = np.load(p, allow_pickle=True).item()
#             cam_stack.append(cam_item)

#         greyscale_heatmap.append(cam_stack)

#     return greyscale_heatmap

def ifcnn_gradcam():
    model = myIFCNN(w0=0.7, w1=0.3)
    model.load_state_dict(torch.load('weights/IFCNN-SUM.pth', map_location=torch.device('cpu')))
    model.eval()
    
    for a in ["train", "test"]:
        resnet_files = sorted(Path(f"./Heatmap_{a}/resnet").glob("*.npy"))
        swin_files = sorted(Path(f"./Heatmap_{a}/swinT").glob("*.npy"))
        # vit_files = sorted(Path(f"./Heatmap_{a}/viT").glob("*.npy"))

        for j in range(len(resnet_files)):
            item1 = np.load(resnet_files[j], allow_pickle=True).item()
            item2 = np.load(swin_files[j], allow_pickle=True).item()
            # item3 = np.load(vit_files[j], allow_pickle=True).item()
            
            t_resnet = torch.tensor(item1.to_heatmap().copy(), dtype=torch.float32).permute(2, 0, 1).unsqueeze(0) 
            t_swint = torch.tensor(item2.to_heatmap().copy(), dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)
            # t_vit = torch.tensor(item3.to_heatmap().copy(), dtype=torch.float32).permute(2, 0, 1).unsqueeze(0) 
            
            with torch.no_grad():
                out = model(t_resnet, t_swint)
            greyscale_cam_IFCNN = show_ifcnn_heatmap(out)
            
            img_path = Path("./Dataset") / item1.label / item1.name
            original_img = Image.open(img_path).convert("RGB").resize(target_size) 
            
            cam_item = GrayscaleCAMItem(greyscale_cam_IFCNN, item1.name, item1.label, original_img)
            save_dir = Path(f"./Heatmap_{a}/resnet+swinT")
            save_dir.mkdir(parents=True, exist_ok=True)
            np.save(save_dir / f"{j}", cam_item)
            if j % 100 == 0:
                print(f"Processed {j}/{len(resnet_files)} images")
            
            
        # axs = plt.figure(figsize=(15, 5)).subplots(2, 4)
        # axs = axs.flatten()
        # axs[0].imshow(item1.to_heatmap())
        # axs[0].set_title("resnet152_gradcam")
        # axs[0].axis("off")
        # axs[1].imshow(item2.to_heatmap())
        # axs[1].set_title("swinT_gradcam")
        # axs[1].axis("off")
        # axs[2].imshow(item3.to_heatmap())
        # axs[2].set_title("viT_gradcam")
        # axs[2].axis("off")
        # axs[3].imshow(greyscale_cam_IFCNN, cmap='jet')
        # axs[3].set_title("IFCNN")
        # axs[3].axis("off")
        # axs[4].imshow(original_img)
        # axs[4].set_title("Original Image")
        # axs[4].axis("off")
        # axs[5].imshow(cam_item.to_mask(), cmap='gray')
        # axs[5].set_title("IFCNN Mask")
        # axs[5].axis("off")
        # axs[6].imshow(cam_item.to_applied_mask(threshold=0.4))
        # axs[6].set_title("IFCNN Applied Mask")
        # axs[6].axis("off")
        
        # plt.show()

In [ ]:
ifcnn_gradcam()

SVM

In [5]:
from sklearn.svm import SVC
from sklearn.decomposition import IncrementalPCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

In [12]:
# Assuming your data is loaded into the following variables:
# X_train, X_test (Image arrays)
# y_train, y_test (Labels: 0 to 4)

def train_image_svm(X_train, y_train, X_test, y_test):
    # 1. Flatten the images
    # Reshape from (num_samples, height, width, channels) to (num_samples, features)
    
    # 2. Scale the data
    # SVMs are highly sensitive to feature scaling; this centers around 0 with a variance of 1.
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 3. Initialize and train the SVM
    # The default kernel is 'rbf'. For 5 classes, sklearn automatically uses One-vs-One (OvO).
    print("Training SVM... this might take a while depending on dataset size.")
    svm_model = SVC(kernel='rbf', C=1.0, random_state=42)
    svm_model.fit(X_train_scaled, y_train)
    
    # 4. Predict and evaluate
    y_pred = svm_model.predict(X_test_scaled)
    
    print("\nAccuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    
    return svm_model, scaler

# Example usage:
# model, fitted_scaler = train_image_svm(X_train, y_train, X_test, y_test)

In [11]:
files_train = sorted(Path("./Heatmap_train/resnet+swinT+viT").glob("*.npy"))
img_train = np.empty((len(files_train), 224 * 224 * 3), dtype=np.float32)
labels_train = np.empty(len(files_train), dtype=object)  # Assuming labels are strings or categorical

for j in range(len(files_train)):
    item1 = np.load(files_train[j], allow_pickle=True).item()
    img_train[j] = np.array(item1.to_applied_mask(threshold=0.4)).flatten()
    labels_train[j] = item1.label  # Assuming each item has a 'label' key
    
files_test = sorted(Path("./Heatmap_test/resnet+swinT+viT").glob("*.npy"))
img_test = np.empty((len(files_test), 224 * 224 * 3), dtype=np.float32)
labels_test = np.empty(len(files_test), dtype=object)  # Assuming labels are strings or categorical

for j in range(len(files_test)):
    item1 = np.load(files_test[j], allow_pickle=True).item()
    img_test[j] = np.array(item1.to_applied_mask(threshold=0.4)).flatten()
    labels_test[j] = item1.label  # Assuming each item has a 'label' key



In [6]:
import joblib

In [13]:
# X_train_flat = X_train.reshape(np.array(img_train).shape[0], -1).astype(np.float32)
# X_test_flat = X_test.reshape(np.array(img_test).shape[0], -1).astype(np.float32)


# print("Running Incremental PCA in batches...")
# # The batch_size dictates how many images it processes at once. 
# # Lower it (e.g., to 128) if you still get a MemoryError.
# ipca = IncrementalPCA(n_components=500, batch_size=256) 

# X_train_pca = ipca.fit_transform(X_train_flat)
# X_test_pca = ipca.transform(X_test_flat)

model_res_vit_swint, scaler_res_vit_swint = train_image_svm(
    img_train,
    labels_train,
    img_test,
    labels_test
)
print("SVM model and scaler trained successfully.")

joblib.dump(
    {"model": model_res_vit_swint, "scaler": scaler_res_vit_swint},
    "svm_res_vit_swint.joblib"
)

Training SVM... this might take a while depending on dataset size.

Accuracy: 0.4303370786516854

Classification Report:
               precision    recall  f1-score   support

   Butterfly       0.37      0.40      0.38       190
   Dragonfly       0.47      0.44      0.45       210
 Grasshopper       0.36      0.51      0.42       182
    Ladybird       0.54      0.43      0.48       163
    Mosquito       0.54      0.35      0.43       145

    accuracy                           0.43       890
   macro avg       0.46      0.43      0.43       890
weighted avg       0.45      0.43      0.43       890

SVM model and scaler trained successfully.


['svm_res_vit_swint.joblib']

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def run_all_models():
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = [
            executor.submit(long_execution1),
            executor.submit(long_execution2),
            executor.submit(long_execution3),
        ]
        for future in futures:
            future.result()

run_all_models()
ifcnn_gradcam()